# HBL BiLSTM — Improved Regression Model v2
**Dataset:** HBL OHLCV + Dawn News Sentiment | 2016-01-27 → 2025-08-21 (2,479 rows)  
**Model:** Bidirectional LSTM (Regression) — Improved with 5 fixes over v1

### Improvements over v1
| Fix | What changed | Why |
|-----|-------------|-----|
| ✅ Directional loss | MSE + sign-penalty replaces plain MSE | MSE rewarded predicting ~0; model now penalised for wrong direction |
| ✅ Dual output head | Auxiliary 5-day forward return head added | Forces LSTM to encode medium-term trend; regularises 1-day head |
| ✅ Sentiment re-engineering | Sparse raw features → surprise, momentum, tone | 91.6% of days have zero news; raw averages just propagate zeros |
| ✅ Conviction threshold | Signal only when pred > tuned threshold | Filters near-zero low-confidence predictions |
| ✅ Full retrain | New scaler + new model weights | All changes require training from scratch |


## 0. Setup & Imports

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#3a3d4d',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2a2d3d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})
ACCENT  = '#00c8ff'
ACCENT2 = '#ff6b6b'
ACCENT3 = '#7bed9f'

print('TensorFlow :', tf.__version__)
print('Setup complete ✓')


TensorFlow : 2.20.0
Setup complete ✓


## 1. Load Data & Feature Engineering

### Key dataset facts (discovered from EDA)
- **2,503 rows** total; **2,479 usable** after rolling-window NaN drop  
- **91.6 % of days have zero news** (`daily_news_count == 0`)  
- Raw `sent_3` / `sent_7` rolling means were mostly flat zeros → negative permutation importance in v1  
- New features target the *rare non-zero news days* as events, not averages


In [8]:
# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv("hbl_merged_dataset_clean.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# ── Price / technical features (same as v1) ───────────────────────────────────
df["ret_1"]  = df["log_return"]
df["ret_5"]  = np.log(df["Close"]).diff(5)
df["ret_10"] = np.log(df["Close"]).diff(10)

df["vol_5"]  = df["log_return"].rolling(5).std()
df["vol_10"] = df["log_return"].rolling(10).std()
df["vol_20"] = df["log_return"].rolling(20).std()

df["hl_pct"]     = (df["High"] - df["Low"]) / df["Close"]
df["oc_pct"]     = (df["Close"] - df["Open"]) / df["Open"]
df["log_volume"] = np.log1p(df["Volume"])

# ── IMPROVED sentiment features ───────────────────────────────────────────────
# Context: 91.6% of days have daily_news_count == 0 and daily_sentiment == 0
# Raw rolling means (sent_3, sent_7) just average zeros → no signal
# Strategy: treat non-zero news days as events; encode surprise and tone

# 1. Sentiment surprise: how much does today's sentiment deviate from recent baseline?
#    On zero-news days this is 0 - rolling_mean ≈ 0; on news days it fires properly.
df["sent_surprise"] = (
    df["daily_sentiment"] - df["daily_sentiment"].rolling(20, min_periods=5).mean()
)

# 2. Sentiment momentum: is the (sparse) sentiment signal trending up or down?
df["sent_momentum"] = df["daily_sentiment"].diff(3)

# 3. Sentiment tone: positive count minus negative count (raw article counts)
#    Gives a directional tone signal independent of the normalised sentiment score.
df["sent_tone"] = df["daily_pos"] - df["daily_neg"]

# 4. Weighted sentiment: sentiment × log(1 + news_count)
#    Amplifies signal on rare high-count days; stays near 0 on zero-news days.
df["sent_weighted"] = df["daily_sentiment"] * np.log1p(df["daily_news_count"])

# 5. News shock: same formula as v1 (kept — had small positive importance)
df["news_shock"] = df["daily_sentiment"] * np.log1p(df["daily_news_count"])

# ── Auxiliary target: 5-day forward log return (Step 3) ──────────────────────
# Used only for the auxiliary output head during training.
# Provides medium-term trend signal to the shared LSTM layers.
df["log_return_5d_fwd"] = np.log(df["Close"]).shift(-5) - np.log(df["Close"])

df = df.dropna().reset_index(drop=True)

# ── Final feature list ────────────────────────────────────────────────────────
# Dropped from v1: sent_3, sent_7 (negative permutation importance)
# Dropped from v1: ret_5 (negative permutation importance)
# Added: sent_surprise, sent_momentum, sent_tone, sent_weighted
FEATURES = [
    # Price momentum
    "ret_1", "ret_10",
    # Volatility (top features in v1)
    "vol_5", "vol_10", "vol_20",
    # Intraday structure
    "hl_pct", "oc_pct",
    # Volume
    "log_volume",
    # Sentiment — re-engineered
    "daily_sentiment",       # raw (kept as anchor)
    "daily_news_count",      # raw count
    "sent_surprise",         # deviation from 20d baseline
    "sent_momentum",         # 3-day trend
    "sent_tone",             # pos - neg article counts
    "sent_weighted",         # sentiment × log(count)
    # Longer-horizon sentiment (had small positive importance in v1)
    "quarterly_sentiment",
    "annual_sentiment",
]

TARGET_1D = "log_return_next"
TARGET_5D = "log_return_5d_fwd"

X    = df[FEATURES].values.astype(np.float32)
y_1d = df[TARGET_1D].values.astype(np.float32)
y_5d = df[TARGET_5D].values.astype(np.float32)
dates = df["date"].values

print(f"Dataset rows  : {len(df):,}  ({df['date'].min().date()} → {df['date'].max().date()})")
print(f"Features      : {len(FEATURES)}")
print(f"1-day target  : range = [{y_1d.min():.4f}, {y_1d.max():.4f}]  std={y_1d.std():.4f}")
print(f"5-day target  : range = [{y_5d.min():.4f}, {y_5d.max():.4f}]  std={y_5d.std():.4f}")
print()
print("Feature list:")
for i, f in enumerate(FEATURES):
    print(f"  {i+1:>2}. {f}")


Dataset rows  : 2,479  (2016-01-27 → 2025-08-21)
Features      : 16
1-day target  : range = [-0.1559, 0.1357]  std=0.0192
5-day target  : range = [-0.2930, 0.2741]  std=0.0489

Feature list:
   1. ret_1
   2. ret_10
   3. vol_5
   4. vol_10
   5. vol_20
   6. hl_pct
   7. oc_pct
   8. log_volume
   9. daily_sentiment
  10. daily_news_count
  11. sent_surprise
  12. sent_momentum
  13. sent_tone
  14. sent_weighted
  15. quarterly_sentiment
  16. annual_sentiment


## 2. Train / Val / Test Split & Scaling
**Split:** 70% train | 15% val | 15% test  
Expected after sequence trimming: ~1,705 train | ~342 val | ~342 test sequences


In [10]:
n = len(df)
train_end = int(n * 0.70)   # ≈ 1,735
val_end   = int(n * 0.85)   # ≈ 2,107

X_train = X[:train_end];       y1_train = y_1d[:train_end]; y5_train = y_5d[:train_end]
X_val   = X[train_end:val_end]; y1_val = y_1d[train_end:val_end]; y5_val = y_5d[train_end:val_end]
X_test  = X[val_end:];         y1_test  = y_1d[val_end:];  y5_test  = y_5d[val_end:]

dates_train = dates[:train_end]
dates_val   = dates[train_end:val_end]
dates_test  = dates[val_end:]

print(f"Raw split sizes — Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")
print(f"Train date range : {pd.to_datetime(dates_train[0]).date()} → {pd.to_datetime(dates_train[-1]).date()}")
print(f"Val date range   : {pd.to_datetime(dates_val[0]).date()} → {pd.to_datetime(dates_val[-1]).date()}")
print(f"Test date range  : {pd.to_datetime(dates_test[0]).date()} → {pd.to_datetime(dates_test[-1]).date()}")

# Fit scaler ONLY on training data to prevent data leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, "feature_scaler_v2.pkl")
print("\nScaler saved → feature_scaler_v2.pkl")

# ── Sequence generation ───────────────────────────────────────────────────────
SEQ_LEN = 30

def make_sequences(X, y1, y5, dates, seq_len=30):
    Xs, y1s, y5s, ds = [], [], [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i - seq_len:i])
        y1s.append(y1[i])
        y5s.append(y5[i])
        ds.append(dates[i])
    return (np.array(Xs, dtype=np.float32),
            np.array(y1s, dtype=np.float32),
            np.array(y5s, dtype=np.float32),
            np.array(ds))

Xtr, y1tr, y5tr, dtr = make_sequences(X_train_sc, y1_train, y5_train, dates_train, SEQ_LEN)
Xva, y1va, y5va, dva = make_sequences(X_val_sc,   y1_val,   y5_val,   dates_val,   SEQ_LEN)
Xte, y1te, y5te, dte = make_sequences(X_test_sc,  y1_test,  y5_test,  dates_test,  SEQ_LEN)

print(f"\nSequence shapes — Train: {Xtr.shape}  Val: {Xva.shape}  Test: {Xte.shape}")


Raw split sizes — Train: 1,735  Val: 372  Test: 372
Train date range : 2016-01-27 → 2022-09-21
Val date range   : 2022-09-22 → 2024-02-26
Test date range  : 2024-02-27 → 2025-08-21

Scaler saved → feature_scaler_v2.pkl

Sequence shapes — Train: (1705, 30, 16)  Val: (342, 30, 16)  Test: (342, 30, 16)


## 3. Custom Directional Loss

**Why MSE failed in v1:**  
On a zero-mean, noisy return series, MSE is minimised by predicting ≈0 always.  
The model was mathematically correct on average but useless in practice (Pearson r = 0.017).

**What directional loss does:**  
Adds an extra penalty proportional to `relu(-y_true × y_pred)`:
- Signs agree → product > 0 → relu = 0 → **no extra penalty**  
- Signs differ → product < 0 → relu fires → **extra penalty proportional to how wrong**

`alpha = 0.5` means the directional penalty is half the MSE weight.  
Increase to `1.0` if directional accuracy is still stuck near 50% after training.


In [11]:
def directional_loss(y_true, y_pred):
    """
    Combined MSE + directional penalty loss.
    
    Parameters
    ----------
    alpha : float
        Weight of directional penalty relative to MSE.
        0.5 = balanced start; increase to 1.0 if direction still poor.
    """
    alpha = 0.5
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    # relu(-y_true * y_pred): fires only when predicted sign is wrong
    direction_penalty = tf.reduce_mean(tf.nn.relu(-y_true * y_pred))
    return mse + alpha * direction_penalty

# ── Sanity check ──────────────────────────────────────────────────────────────
y_t = tf.constant([ 0.010, -0.020,  0.005, -0.008], dtype=tf.float32)

# All correct signs
y_p_good = tf.constant([ 0.008, -0.015,  0.003, -0.005], dtype=tf.float32)
# Two wrong signs
y_p_bad  = tf.constant([ 0.008,  0.015,  0.003,  0.005], dtype=tf.float32)

loss_good = directional_loss(y_t, y_p_good).numpy()
loss_bad  = directional_loss(y_t, y_p_bad).numpy()
mse_good  = tf.reduce_mean(tf.square(y_t - y_p_good)).numpy()
mse_bad   = tf.reduce_mean(tf.square(y_t - y_p_bad)).numpy()

print(f"Correct signs  → MSE={mse_good:.8f}  DirectionalLoss={loss_good:.8f}")
print(f"2 wrong signs  → MSE={mse_bad:.8f}   DirectionalLoss={loss_bad:.8f}")
print(f"Penalty for wrong signs: +{loss_bad - loss_good:.8f}")
print("Custom loss function ready ✓")


Correct signs  → MSE=0.00001050  DirectionalLoss=0.00001050
2 wrong signs  → MSE=0.00035050   DirectionalLoss=0.00039300
Penalty for wrong signs: +0.00038250
Custom loss function ready ✓


## 4. Build Improved BiLSTM — Dual Output Head

**Architecture (same BiLSTM stack as v1, new heads):**
```
Input (30, 16)
  → BiLSTM(128, return_seq=True) + BatchNorm
  → BiLSTM(64,  return_seq=False) + BatchNorm
  → Dense(64, relu) → Dropout(0.3) → Dense(32, relu)   ← shared trunk
       ├─ Head 1: Dense(16) → Dense(1)  →  pred_1d   [directional loss, weight=1.0]
       └─ Head 2: Dense(16) → Dense(1)  →  pred_5d   [MSE,              weight=0.3]
```
The 5-day head is an **auxiliary regulariser** — it forces the LSTM to encode  
medium-term trend. Its low weight (0.3) guides without dominating the 1-day objective.


In [12]:
def build_improved_bilstm(seq_len, n_features):
    inp = layers.Input(shape=(seq_len, n_features), name="input")

    # ── Bidirectional LSTM stack ──────────────────────────────────────────────
    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1),
        name="bilstm_1"
    )(inp)
    x = layers.BatchNormalization(name="bn_1")(x)

    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.1),
        name="bilstm_2"
    )(x)
    x = layers.BatchNormalization(name="bn_2")(x)

    # ── Shared dense trunk ────────────────────────────────────────────────────
    shared = layers.Dense(64, activation="relu", name="dense_shared")(x)
    shared = layers.Dropout(0.3, name="dropout_shared")(shared)
    shared = layers.Dense(32, activation="relu", name="dense_shared_2")(shared)

    # ── Head 1: 1-day return (primary, directional loss) ─────────────────────
    h1 = layers.Dense(16, activation="relu", name="head1_dense")(shared)
    out_1d = layers.Dense(1, name="pred_1d")(h1)

    # ── Head 2: 5-day return (auxiliary, MSE) ────────────────────────────────
    h2 = layers.Dense(16, activation="relu", name="head2_dense")(shared)
    out_5d = layers.Dense(1, name="pred_5d")(h2)

    model = Model(inputs=inp, outputs=[out_1d, out_5d], name="ImprovedBiLSTM_v2")
    return model

model = build_improved_bilstm(SEQ_LEN, len(FEATURES))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "pred_1d": directional_loss,   # primary: directional loss
        "pred_5d": "mse",              # auxiliary: plain MSE is fine
    },
    loss_weights={
        "pred_1d": 1.0,
        "pred_5d": 0.3,   # auxiliary guides, does not dominate
    },
    metrics={
        "pred_1d": ["mae"],
        "pred_5d": ["mae"],
    },
)

model.summary()
print(f"\nTotal params: {model.count_params():,}")
print("Model built ✓")


Model: "ImprovedBiLSTM_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)            │ (None, 30, 16)            │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bilstm_1 (Bidirectional)      │ (None, 30, 256)           │         148,480 │ input[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_1 (BatchNormalization)     │ (None, 30, 256)           │           1,024 │ bilstm_1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bilstm_2 (Bidirectional)      │ (None, 128)               │         164,352 │ bn_1[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_2 (BatchNormalization)     │ (None, 128)               │             512 │ bilstm_2[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_shared (Dense)          │ (None, 64)                │           8,256 │ bn_2[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_shared (Dropout)      │ (None, 64)                │               0 │ dense_shared[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_shared_2 (Dense)        │ (None, 32)                │           2,080 │ dropout_shared[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ head1_dense (Dense)           │ (None, 16)                │             528 │ dense_shared_2[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ head2_dense (Dense)           │ (None, 16)                │             528 │ dense_shared_2[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pred_1d (Dense)               │ (None, 1)                 │              17 │ head1_dense[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pred_5d (Dense)               │ (None, 1)                 │              17 │ head2_dense[0][0]          │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 325,794 (1.24 MB)

 Trainable params: 325,026 (1.24 MB)

 Non-trainable params: 768 (3.00 KB)


Total params: 325,794
Model built ✓


## 5. Train the Model
**Expected training time:** ~5-15 minutes depending on hardware  
**EarlyStopping** monitors `val_pred_1d_loss` with patience=20  
**ReduceLROnPlateau** halves LR after 8 epochs of no improvement


In [13]:
cb_early = callbacks.EarlyStopping(
    monitor="val_pred_1d_loss",
    mode="min",
    patience=20,
    restore_best_weights=True,
    verbose=1
)
cb_reduce = callbacks.ReduceLROnPlateau(
    monitor="val_pred_1d_loss",
    mode="min",
    factor=0.5,
    patience=8,
    min_lr=1e-6,
    verbose=1
)
cb_save = callbacks.ModelCheckpoint(
    "bilstm_improved_v2.keras",
    monitor="val_pred_1d_loss",
    mode="min",
    save_best_only=True,
    verbose=0
)

## 6. Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training History — Improved BiLSTM v2", fontsize=13, fontweight="bold")

# 1-day head loss
ax = axes[0]
ax.plot(history.history["pred_1d_loss"],     color=ACCENT,  lw=1.5, label="Train")
ax.plot(history.history["val_pred_1d_loss"], color=ACCENT2, lw=1.5, label="Val")
ax.set_title("1-Day Head Loss (Directional)"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, alpha=0.3)

# 5-day head loss
ax = axes[1]
ax.plot(history.history["pred_5d_loss"],     color=ACCENT,  lw=1.5, label="Train")
ax.plot(history.history["val_pred_5d_loss"], color=ACCENT2, lw=1.5, label="Val")
ax.set_title("5-Day Head Loss (MSE)"); ax.set_xlabel("Epoch"); ax.set_ylabel("MSE")
ax.legend(); ax.grid(True, alpha=0.3)

# 1-day MAE
ax = axes[2]
ax.plot(history.history["pred_1d_mae"],     color=ACCENT,  lw=1.5, label="Train")
ax.plot(history.history["val_pred_1d_mae"], color=ACCENT2, lw=1.5, label="Val")
ax.set_title("1-Day MAE"); ax.set_xlabel("Epoch"); ax.set_ylabel("MAE")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("plot_v2_training_history.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved → plot_v2_training_history.png")


## 7. Generate Predictions on All Splits

In [15]:
# Reload best model weights
model = tf.keras.models.load_model(
    "bilstm_return_regression.keras",
    custom_objects={"directional_loss": directional_loss}
)

def get_1d_preds(model, X):
    """Extract 1-day head predictions from dual-output model."""
    out = model.predict(X, verbose=0)
    # out[0] = pred_1d, out[1] = pred_5d
    return out[0].flatten()

pred_train = get_1d_preds(model, Xtr)
pred_val   = get_1d_preds(model, Xva)
pred_test  = get_1d_preds(model, Xte)

print("Predictions generated ✓")
print(f"Test pred range   : [{pred_test.min():.5f}, {pred_test.max():.5f}]")
print(f"Test actual range : [{y1te.min():.5f}, {y1te.max():.5f}]")
print()
pred_spread = pred_test.max() - pred_test.min()
actual_spread = y1te.max() - y1te.min()
print(f"Prediction spread vs actual spread: {pred_spread:.5f} vs {actual_spread:.5f}")
print(f"  (v1 had ~0.017 pred spread vs 0.17 actual — model was near-flat)")
print(f"  (v2 target: meaningfully wider spread)")


Predictions generated ✓
Test pred range   : [-0.01359, -0.01359]
Test actual range : [-0.07806, 0.09532]

Prediction spread vs actual spread: 0.00000 vs 0.17338
  (v1 had ~0.017 pred spread vs 0.17 actual — model was near-flat)
  (v2 target: meaningfully wider spread)


## 8. Evaluation Metrics — All Splits

In [16]:
def compute_metrics(actual, predicted, split_name):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae  = mean_absolute_error(actual, predicted)
    r2   = r2_score(actual, predicted)
    corr = np.corrcoef(actual, predicted)[0, 1]
    da   = np.mean(np.sign(actual) == np.sign(predicted)) * 100
    return {
        "Split":      split_name,
        "RMSE":       round(rmse, 6),
        "MAE":        round(mae, 6),
        "R²":         round(r2, 4),
        "Pearson r":  round(corr, 4),
        "Dir. Acc %": round(da, 2),
    }

metrics = pd.DataFrame([
    compute_metrics(y1tr, pred_train, "Train"),
    compute_metrics(y1va, pred_val,   "Validation"),
    compute_metrics(y1te, pred_test,  "Test"),
]).set_index("Split")

print("\n── Evaluation Metrics (v2 Improved) ─────────────────────────────")
print(metrics.to_string())
print()
print("── v1 Baseline (for comparison) ──────────────────────────────────")
print("   Train R²=0.013  Val R²=0.005  Test R²=-0.041  Test Dir.Acc=51.6%  Pearson r=0.017")
print()

r2_test = metrics.loc["Test", "R²"]
da_test = metrics.loc["Test", "Dir. Acc %"]
r_test  = metrics.loc["Test", "Pearson r"]

print(f"📌 Test R²        : {r2_test:.4f}  (v1: -0.041)")
print(f"📌 Test Pearson r : {r_test:.4f}  (v1:  0.017)")
print(f"📌 Test Dir. Acc  : {da_test:.1f}%  (v1: 51.6%)")
print()
if da_test > 53:
    print("✅ Directional accuracy meaningfully above random — improvement confirmed")
elif da_test > 51.6:
    print("⚠️  Marginal improvement in direction. Try alpha=1.0 in directional_loss and retrain.")
else:
    print("⚠️  Direction not yet improved. See tuning tips in Cell 14.")


ValueError: Found input variables with inconsistent numbers of samples: [1705, 1]

## 9. Conviction Threshold Tuning (Step 5)
Only trade when the model predicts with **conviction above threshold X**.  
Tune X on the **validation set** (never on test) to maximise Sharpe ratio.  
This filters out near-zero low-confidence predictions that are mostly noise.


In [ ]:
def sharpe_at_threshold(pred, actual, threshold):
    """Long-only strategy Sharpe: LONG if pred > threshold, else FLAT."""
    signal = np.where(pred > threshold, 1, 0)
    strat_rets = signal * actual
    if strat_rets.std() < 1e-9:
        return 0.0
    return (strat_rets.mean() / strat_rets.std()) * np.sqrt(252)

# Sweep thresholds on validation set
thresholds  = np.linspace(0.0, 0.015, 151)
sharpes_val = [sharpe_at_threshold(pred_val, y1va, t) for t in thresholds]

best_idx       = int(np.argmax(sharpes_val))
BEST_THRESHOLD = thresholds[best_idx]
best_sharpe_val = sharpes_val[best_idx]

print(f"Optimal conviction threshold (val): {BEST_THRESHOLD:.5f}")
print(f"Val Sharpe at threshold=0.00000   : {sharpe_at_threshold(pred_val, y1va, 0.0):.3f}")
print(f"Val Sharpe at threshold={BEST_THRESHOLD:.5f}  : {best_sharpe_val:.3f}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(thresholds, sharpes_val, color=ACCENT, lw=2)
ax.axvline(BEST_THRESHOLD, color=ACCENT2, lw=2, linestyle="--",
           label=f"Best threshold = {BEST_THRESHOLD:.5f}  (Sharpe={best_sharpe_val:.2f})")
ax.axhline(sharpe_at_threshold(pred_val, y1va, 0.0), color="white", lw=1,
           linestyle=":", alpha=0.6, label="Threshold = 0 (v1 approach)")
ax.set_xlabel("Conviction Threshold"); ax.set_ylabel("Validation Sharpe")
ax.set_title("Threshold Tuning on Validation Set")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("plot_v2_threshold_tuning.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

# Trade counts on test set
n_long = int(np.sum(pred_test > BEST_THRESHOLD))
n_flat = len(pred_test) - n_long
print(f"\nTest set: {n_long} LONG ({n_long/len(pred_test)*100:.0f}%)  |  {n_flat} FLAT ({n_flat/len(pred_test)*100:.0f}%)")


## 10. Actual vs Predicted — Test Set

In [ ]:
test_dates = pd.to_datetime(dte)

fig, axes = plt.subplots(3, 1, figsize=(16, 13),
                          gridspec_kw={"height_ratios": [3, 1.2, 1.2]})
fig.suptitle("HBL BiLSTM v2 — Actual vs Predicted Log Returns (Test Set)",
             fontsize=15, fontweight="bold", y=1.01)

ax = axes[0]
ax.plot(test_dates, y1te,      color=ACCENT,  lw=1.2, alpha=0.9, label="Actual")
ax.plot(test_dates, pred_test, color=ACCENT2, lw=1.0, alpha=0.8, label="Predicted v2", linestyle="--")
ax.axhline(0, color="white", lw=0.5, alpha=0.3)
ax.set_title("Log Return: Actual vs Predicted"); ax.set_ylabel("Log Return")
ax.legend(); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

residuals = y1te - pred_test
ax2 = axes[1]
ax2.bar(test_dates, residuals,
        color=np.where(residuals >= 0, ACCENT3, ACCENT2), alpha=0.7, width=1.5)
ax2.axhline(0, color="white", lw=0.8)
ax2.set_title("Residuals (Actual − Predicted)"); ax2.set_ylabel("Residual")
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax2.get_xticklabels(), rotation=30, ha="right")

correct  = (np.sign(y1te) == np.sign(pred_test)).astype(float)
roll_da  = pd.Series(correct).rolling(30).mean() * 100

ax3 = axes[2]
ax3.plot(test_dates, roll_da, color=ACCENT3, lw=1.2)
ax3.axhline(50, color=ACCENT2, lw=1, linestyle="--", alpha=0.8, label="Random baseline (50%)")
ax3.fill_between(test_dates, 50, roll_da, where=(roll_da >= 50), color=ACCENT3, alpha=0.2)
ax3.fill_between(test_dates, 50, roll_da, where=(roll_da <  50), color=ACCENT2, alpha=0.2)
ax3.set_title("Rolling 30-Day Directional Accuracy"); ax3.set_ylabel("Accuracy %")
ax3.set_ylim(20, 90); ax3.legend(); ax3.grid(True, alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax3.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig("plot_v2_actual_vs_predicted.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


## 11. Residual Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Residual Analysis — Test Set (v2)", fontsize=14, fontweight="bold")

residuals = y1te - pred_test
r_val = np.corrcoef(y1te, pred_test)[0, 1]

ax = axes[0, 0]
ax.scatter(y1te, pred_test, color=ACCENT, alpha=0.4, s=12, edgecolors="none")
lims = [min(y1te.min(), pred_test.min()), max(y1te.max(), pred_test.max())]
ax.plot(lims, lims, color=ACCENT2, lw=1.5, linestyle="--", label="Perfect prediction")
ax.set_xlabel("Actual Log Return"); ax.set_ylabel("Predicted Log Return")
ax.set_title(f"Actual vs Predicted  (r = {r_val:.3f}  |  v1: 0.017)")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.hist(residuals, bins=60, color=ACCENT, alpha=0.75, edgecolor="none", density=True)
mu, sigma = residuals.mean(), residuals.std()
xr = np.linspace(residuals.min(), residuals.max(), 200)
ax.plot(xr, stats.norm.pdf(xr, mu, sigma), color=ACCENT2, lw=2,
        label=f"Normal(μ={mu:.4f}, σ={sigma:.4f})")
ax.set_xlabel("Residual"); ax.set_ylabel("Density")
ax.set_title("Residual Distribution"); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
(osm, osr), (slope, intercept, _) = stats.probplot(residuals, dist="norm")
ax.scatter(osm, osr, color=ACCENT, alpha=0.5, s=10, edgecolors="none")
ax.plot(osm, slope * np.array(osm) + intercept, color=ACCENT2, lw=2, linestyle="--")
ax.set_xlabel("Theoretical Quantiles"); ax.set_ylabel("Sample Quantiles")
ax.set_title("Q-Q Plot"); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.scatter(pred_test, residuals, color=ACCENT3, alpha=0.4, s=12, edgecolors="none")
ax.axhline(0, color=ACCENT2, lw=1.5, linestyle="--")
ax.set_xlabel("Predicted Log Return"); ax.set_ylabel("Residual")
ax.set_title("Predicted vs Residuals (Heteroscedasticity Check)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("plot_v2_residual_analysis.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


## 12. Strategy Backtest — v1 vs v2 vs Buy & Hold
Three strategies compared on the test period:
- **v2 (threshold):** LONG only when pred > `BEST_THRESHOLD` (tuned on val)  
- **v2 (pred > 0):**  same signals as v1 but with new model weights  
- **Buy & Hold:**     always LONG


In [ ]:
signal_v2_thresh = np.where(pred_test > BEST_THRESHOLD, 1, 0)
signal_v2_zero   = np.where(pred_test > 0, 1, 0)
bh_ret           = y1te

strat_v2_t = signal_v2_thresh * y1te
strat_v2_0 = signal_v2_zero   * y1te

cum_v2_t = np.exp(np.cumsum(strat_v2_t)) - 1
cum_v2_0 = np.exp(np.cumsum(strat_v2_0)) - 1
cum_bh   = np.exp(np.cumsum(bh_ret))     - 1

def sharpe(rets):
    return (rets.mean() / (rets.std() + 1e-9)) * np.sqrt(252)

def max_dd(cum_ret):
    wealth   = 1 + cum_ret
    roll_max = np.maximum.accumulate(wealth)
    return ((wealth - roll_max) / roll_max * 100)

sh_v2_t = sharpe(strat_v2_t)
sh_v2_0 = sharpe(strat_v2_0)
sh_bh   = sharpe(bh_ret)
dd_v2_t = max_dd(cum_v2_t)
dd_bh   = max_dd(cum_bh)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("Strategy Backtest — Test Period", fontsize=14, fontweight="bold")

ax = axes[0]
ax.plot(test_dates, cum_v2_t * 100, color=ACCENT3, lw=2.5,
        label=f"v2 w/ threshold={BEST_THRESHOLD:.4f}  (Sharpe={sh_v2_t:.2f})")
ax.plot(test_dates, cum_v2_0 * 100, color="#ffa502", lw=1.5, linestyle="--",
        label=f"v2 pred>0                (Sharpe={sh_v2_0:.2f})", alpha=0.8)
ax.plot(test_dates, cum_bh   * 100, color=ACCENT,  lw=1.5,
        label=f"Buy & Hold               (Sharpe={sh_bh:.2f})",   alpha=0.7)
ax.axhline(0, color="white", lw=0.5, alpha=0.4)
ax.fill_between(test_dates, 0, cum_v2_t * 100, where=(cum_v2_t >= 0), color=ACCENT3, alpha=0.1)
ax.fill_between(test_dates, 0, cum_v2_t * 100, where=(cum_v2_t <  0), color=ACCENT2, alpha=0.1)
ax.set_title("Cumulative Return %"); ax.set_ylabel("Cumulative Return (%)")
ax.legend(); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

ax2 = axes[1]
ax2.fill_between(test_dates, dd_v2_t, 0, color=ACCENT3, alpha=0.5,
                 label=f"v2 (threshold)  MaxDD={dd_v2_t.min():.1f}%")
ax2.fill_between(test_dates, dd_bh,   0, color=ACCENT2, alpha=0.3,
                 label=f"Buy & Hold      MaxDD={dd_bh.min():.1f}%")
ax2.set_title("Drawdown %"); ax2.set_ylabel("Drawdown (%)")
ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.setp(ax2.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout()
plt.savefig("plot_v2_backtest.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

print("\n── Backtest Summary ─────────────────────────────────────────────")
print(f"  {'Strategy':<35} {'Return':>8}  {'Sharpe':>8}  {'MaxDD':>8}")
print("  " + "-" * 62)
print(f"  {'v2 (threshold=' + f'{BEST_THRESHOLD:.4f})':<35} {cum_v2_t[-1]*100:>7.1f}%  {sh_v2_t:>8.3f}  {dd_v2_t.min():>7.1f}%")
print(f"  {'v2 (pred > 0)':<35} {cum_v2_0[-1]*100:>7.1f}%  {sh_v2_0:>8.3f}  —")
print(f"  {'Buy & Hold':<35} {cum_bh[-1]*100:>7.1f}%  {sh_bh:>8.3f}  {dd_bh.min():>7.1f}%")
print()
print("  v1 reference: Strategy 86%  Sharpe=2.077  |  B&H 156%  Sharpe=1.885")


## 13. Permutation Feature Importance
Verifies whether the new sentiment features (`sent_surprise`, `sent_momentum`, `sent_tone`)  
have positive importance — unlike `sent_3` / `sent_7` which were **negative** in v1.


In [ ]:
np.random.seed(42)
base_rmse = np.sqrt(mean_squared_error(y1te, pred_test))
importances = {}

for i, feat in enumerate(FEATURES):
    Xte_perm = Xte.copy()
    idx = np.random.permutation(Xte_perm.shape[0])
    Xte_perm[:, :, i] = Xte_perm[idx, :, i]
    perm_preds = get_1d_preds(model, Xte_perm)
    perm_rmse  = np.sqrt(mean_squared_error(y1te, perm_preds))
    importances[feat] = perm_rmse - base_rmse

imp_df = pd.Series(importances).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(11, 7))
colors = [ACCENT3 if v > 0 else ACCENT2 for v in imp_df.values]
bars = ax.barh(imp_df.index, imp_df.values, color=colors, alpha=0.85, edgecolor="none")
ax.axvline(0, color="white", lw=0.8, alpha=0.5)
ax.set_xlabel("RMSE Increase when Feature Permuted")
ax.set_title("Permutation Feature Importance (v2)\n"
             "Green=positive contribution  Red=adding noise", fontsize=12)

for bar, val in zip(bars, imp_df.values):
    ax.text(val + (0.000005 if val >= 0 else -0.000005),
            bar.get_y() + bar.get_height() / 2,
            f"{val:.5f}", va="center",
            ha="left" if val >= 0 else "right",
            fontsize=8, color="#e0e0e0")

plt.tight_layout()
plt.savefig("plot_v2_feature_importance.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

print("\nTop 5 most important features:")
print(pd.Series(importances).sort_values(ascending=False).head(5).to_string())
print()
print("Negative importance features (shuffling them HELPS — consider dropping):")
neg_feats = pd.Series(importances)[pd.Series(importances) < 0]
if len(neg_feats):
    print(neg_feats.sort_values().to_string())
else:
    print("  None — all features contribute positively ✅")


## 14. Export Test Predictions to CSV

In [ ]:
test_df = pd.DataFrame({
    "date":                pd.to_datetime(dte),
    "actual_log_ret":      y1te,
    "predicted_log_ret":   pred_test,
    "residual":            y1te - pred_test,
    "signal_v1":           np.where(pred_test > 0,               "LONG", "FLAT"),
    "signal_v2":           np.where(pred_test > BEST_THRESHOLD,  "LONG", "FLAT"),
    "direction_correct":   np.sign(y1te) == np.sign(pred_test),
})

# Attach Close price for context
close_test = df.loc[val_end + SEQ_LEN:, "Close"].reset_index(drop=True)
if len(close_test) == len(test_df):
    test_df["Close"] = close_test.values

test_df.to_csv("hbl_test_predictions_v2.csv", index=False)
print(f"Saved {len(test_df)} test predictions → hbl_test_predictions_v2.csv")
print()
print(test_df.head(10).to_string(index=False))


## 15. Predict on New / Unseen Data

In [ ]:
def predict_next_return_v2(df_raw, model, scaler, features=FEATURES,
                            seq_len=SEQ_LEN, threshold=None):
    """
    Predict next-day log return using improved BiLSTM v2.

    Parameters
    ----------
    df_raw    : pd.DataFrame — full dataset with all feature engineering applied
    model     : loaded Keras dual-output model
    scaler    : fitted StandardScaler (feature_scaler_v2.pkl)
    features  : list of feature column names (must match FEATURES used in training)
    seq_len   : int — must match training SEQ_LEN (30)
    threshold : float — conviction threshold; if None uses 0 (raw sign)

    Returns
    -------
    dict with prediction details and signal
    """
    assert len(df_raw) >= seq_len, f"Need at least {seq_len} rows."
    assert all(f in df_raw.columns for f in features), \
        "Some features missing from df_raw — run feature engineering first."

    window        = df_raw[features].values[-seq_len:].astype(np.float32)
    window_scaled = scaler.transform(window)
    X_input       = window_scaled[np.newaxis, :, :]   # (1, 30, n_features)

    out         = model.predict(X_input, verbose=0)
    pred_1d     = float(out[0][0, 0])
    pred_5d     = float(out[1][0, 0])
    pred_pct    = (np.exp(pred_1d) - 1) * 100
    direction   = "UP ▲" if pred_1d > 0 else "DOWN ▼"
    thr         = threshold if threshold is not None else 0.0
    signal      = "LONG" if pred_1d > thr else "FLAT"
    conviction  = "HIGH" if abs(pred_1d) > thr * 2 else "LOW"

    last_date   = df_raw["date"].iloc[-1]
    last_close  = df_raw["Close"].iloc[-1]

    return {
        "As of date":                 str(pd.to_datetime(last_date).date()),
        "Last Close (PKR)":           round(last_close, 2),
        "Predicted 1-day log return": round(pred_1d,  6),
        "Predicted 5-day log return": round(pred_5d,  6),
        "Predicted % change (1d)":    round(pred_pct, 3),
        "Predicted next close ~":     round(last_close * np.exp(pred_1d), 2),
        "Direction":                  direction,
        "Signal":                     signal,
        "Conviction":                 conviction,
        "Threshold used":             round(thr, 6),
    }


result = predict_next_return_v2(df, model, scaler, threshold=BEST_THRESHOLD)

print("\n" + "═" * 52)
print("  HBL NEXT-DAY PREDICTION  (BiLSTM v2)")
print("═" * 52)
for k, v in result.items():
    print(f"  {k:<38} {v}")
print("═" * 52)


## 16. Full Timeline — All Splits

In [ ]:
all_dates  = np.concatenate([dtr, dva, dte])
all_actual = np.concatenate([y1tr, y1va, y1te])
all_pred   = np.concatenate([pred_train, pred_val, pred_test])

split_val_date  = pd.to_datetime(dtr[-1])
split_test_date = pd.to_datetime(dva[-1])

fig, ax = plt.subplots(figsize=(18, 6))
all_dt = pd.to_datetime(all_dates)

ax.plot(all_dt, all_actual, color=ACCENT,  lw=0.9, alpha=0.85, label="Actual")
ax.plot(all_dt, all_pred,   color=ACCENT2, lw=0.8, alpha=0.75, label="Predicted v2", linestyle="--")
ax.axvline(split_val_date,  color="yellow", lw=1.5, linestyle=":", alpha=0.8, label="Train│Val")
ax.axvline(split_test_date, color="orange", lw=1.5, linestyle=":", alpha=0.8, label="Val│Test")
ax.axhline(0, color="white", lw=0.4, alpha=0.3)
ax.set_title("HBL BiLSTM v2 — Full Timeline: Actual vs Predicted (2016–2025)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Log Return")
ax.legend(loc="upper left"); ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig("plot_v2_full_timeline.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()


## 17. Final Summary — v1 vs v2

In [ ]:
print("\n" + "═" * 68)
print("  HBL BiLSTM  v1 → v2  IMPROVEMENT SUMMARY")
print("═" * 68)
print()
print("  Changes made:")
changes = {
    "Loss function":      "MSE → Directional loss (MSE + 0.5×sign penalty)",
    "Output heads":       "Single → Dual (1d primary + 5d auxiliary @ 0.3 weight)",
    "Sentiment sent_3/7": "DROPPED (negative importance in v1)",
    "New sent features":  "sent_surprise, sent_momentum, sent_tone, sent_weighted",
    "Signal filter":      f"pred>0 → pred>{BEST_THRESHOLD:.5f} (threshold tuned on val)",
    "Scaler file":        "feature_scaler_v2.pkl  (retrained on same 70% split)",
    "Model file":         "bilstm_improved_v2.keras",
}
for k, v in changes.items():
    print(f"  {'  '+k+':':<30} {v}")

print()
print(f"  {'Metric':<22} {'v1 (baseline)':>16}  {'v2 (test)':>12}")
print("  " + "-" * 54)
v2_r2 = metrics.loc['Test','R²']
v2_r  = metrics.loc['Test','Pearson r']
v2_da = metrics.loc['Test','Dir. Acc %']
print(f"  {'R²':<22} {-0.041:>16.4f}  {v2_r2:>12.4f}")
print(f"  {'Pearson r':<22} {0.017:>16.4f}  {v2_r:>12.4f}")
print(f"  {'Dir. Acc %':<22} {51.6:>16.1f}  {v2_da:>12.1f}")
print()
print("  Tuning tips if results are still poor:")
print("  1. Increase directional_loss alpha: 0.5 → 1.0, retrain")
print("  2. Increase epochs (patience=30) if val loss still declining")
print("  3. Add vol_ratio = vol_5/vol_20 (vol regime feature)")
print("  4. Try SEQ_LEN=60 to give model more historical context")
print("═" * 68)
